In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag.ingestion import (
    ChromaVectorStoreAdmin,
    default_ingestion_run_config
)

resource module not available on Windows


In [2]:
INGESTION_RUN_CONFIG = default_ingestion_run_config(project_root)
COLLECTION_NAME = INGESTION_RUN_CONFIG.collection_name
paths = INGESTION_RUN_CONFIG.paths
CHROMA_DIR = paths.chroma_dir

# Vector database admin

In [3]:
vector_admin = ChromaVectorStoreAdmin(CHROMA_DIR)

In [4]:
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Existing Chroma collections: {vector_admin.list_collection_names()}")

ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Existing Chroma collections: ['news_chat_multilingual_e5_base', 'run_testing']


In [5]:
# Daily vector database operations.
# Run this cell whenever you want to inspect ChromaDB before/after ingestion.
print("Collection count:", vector_admin.count_collections())
print("Collection names:", vector_admin.list_collection_names())

if COLLECTION_NAME in vector_admin.list_collection_names():
    summary = vector_admin.describe_collection(COLLECTION_NAME)
    print("Active collection summary:", summary)
    sample = vector_admin.sample_records(COLLECTION_NAME, limit=2)
    print("Sample ids:", sample.get("ids", []))
    print("Sample metadata:", sample.get("metadatas", []))
else:
    print(f"Collection {COLLECTION_NAME!r} does not exist yet. Run the insertion cell to create it.")

# To delete a collection intentionally, uncomment the next line.
# vector_admin.delete_collection("collection_name_to_delete", missing_ok=True)

Collection count: 2
Collection names: ['news_chat_multilingual_e5_base', 'run_testing']
Active collection summary: ChromaCollectionSummary(name='run_testing', count=1621, metadata={'chunk_size': 800, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'source_folder_count': 1, 'chunk_overlap': 120, 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news'})
Sample ids: ['afd853f0-629c-4d30-a98f-3324e710f16a', '2dad32f8-044d-4db3-b3fd-b96fe4f8099a']
Sample metadata: [{'article_title': 'Chinese homeschooled students embrace freer youth in cutthroat market', 'article_date': '01-01-2026', 'source_folder': 'hk_free_press_news', 'chunk_size': 800, 'chunk_number': 1, 'file_path': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news\\01-01-2026_Chinese homeschooled students embrace freer youth in cutthroat market.txt', 'chunk_overlap': 120, 'source_type': 'news_txt', 'file_name': '01-01-2026_Chinese homeschooled students embrace freer youth 

# Session admin

In [6]:
from src.rag.inference import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_HUGGINGFACE_MODEL_KEY,
    DEFAULT_LLM_PROVIDER,
    DEFAULT_OLLAMA_MODEL,
    create_rag_app,
    default_paths
)

In [7]:
PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
CHROMA_DIR = paths.chroma_dir
SESSION_DIR = paths.session_dir

COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
LLM_PROVIDER = DEFAULT_LLM_PROVIDER  # Choose: LLM_PROVIDER_OLLAMA or LLM_PROVIDER_HUGGINGFACE
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL
HUGGINGFACE_MODEL = DEFAULT_HUGGINGFACE_MODEL_KEY  # deepseek_v3, minimax_m3, qwen_3_5, or any full HF model id
HUGGINGFACE_PROVIDER = "auto"

In [8]:
# It reconnects to ChromaDB, rebuilds semantic + BM25 retrieval, and prepares chat persistence.
rag_app = create_rag_app(
    chroma_dir=CHROMA_DIR,
    session_dir=SESSION_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    llm_provider=LLM_PROVIDER,
    ollama_model=OLLAMA_MODEL,
    huggingface_model=HUGGINGFACE_MODEL,
    huggingface_provider=HUGGINGFACE_PROVIDER,
    final_top_k=5,
)

### Session Management Helpers

Use these APIs to inspect and manage persisted chat sessions:
- `rag_app.count_chat_ids()`
- `rag_app.list_chat_ids()`
- `rag_app.rename_chat(old_chat_id, new_chat_id)`
- `rag_app.delete_chat(chat_id)`

In [9]:
print("Saved chat count:", rag_app.count_chat_ids())
print("Saved chat ids:", rag_app.list_chat_ids())

Saved chat count: 2
Saved chat ids: ['Test_Chat_session', 'Donald_Trump']


In [ ]:
for delete_chat_id in ['chat_20260701_231246', 'chat_20260701_230551', 'chat_20260701_223412']:
    rag_app.delete_chat(delete_chat_id, close_open_session=True, missing_ok=False)